# Triagegeist: Clinical Intensity Prediction in the Emergency Context

### **Authors:** [Amey Thakur](https://www.kaggle.com/ameythakur20) & [Archit Konde](https://www.kaggle.com/architkonde)

***

## 1. Research Objective & Clinical Context

Emergency department (ED) triage constitutes a critical, high-velocity decision environment where clinicians prioritize patient care under significant cognitive load. The **Emergency Severity Index (ESI)** serves as the standard for classifying acuity, ranging from ESI-1 (Immediately life-threatening) to ESI-5 (Least urgent).

This research implements a **Multi-Tier Clinical Decision Support (CDS) System** designed to identify high-acuity patients (ESI-1 and ESI-2) with high precision. The architecture utilizes a synthetic clinical database from the **Laitinen-Fredriksson Foundation**, calibrated to mimic distribution patterns found in the **MIMIC-IV-ED** dataset.

### Core Pipeline Architecture:
1. **Tier 1 (Recognition):** Deterministic pattern mapping of unambiguous chief complaints.
2. **Tier 2 (Specialization):** Targeted diagnostic sub-models for respiratory and hemodynamic distress.
3. **Tier 3 (Generalization):** Balanced gradient-boosted ensemble for systemic acuity classification.

**Citation:** Olaf Yunus Laitinen Imanov (2026). Triagegeist. https://kaggle.com/competitions/triagegeist, 2026. Kaggle.

## 2. Environment Governance

Ensuring deterministic results through seed locking and hardware governance. We integrate `kaggle_toolbox` to manage memory efficiency during high-concurrency relational joins.

In [ ]:
import os
import json
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm.auto import tqdm
from pathlib import Path

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import quadratic_weighted_kappa as qwk_func
from sklearn.metrics import classification_report, confusion_matrix

try:
    import kaggle_toolbox as tb
except ImportError:
    tb = None

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

class CFG:
    SEED = 42
    TARGET = 'triage_acuity'
    N_FOLDS = 5
    # Standard ESI mapping logic
    ESI_MAP = {1: 'ESI-1: Resuscitation', 2: 'ESI-2: Emergent', 
                3: 'ESI-3: Urgent', 4: 'ESI-4: Less Urgent', 5: 'ESI-5: Non-Urgent'}

def seed_everything(seed=42):
    np.random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    if tb: tb.seed_everything(seed)
    
seed_everything(CFG.SEED)
print("Protocol Locked: Computational environment synchronized.")

## 3. Relational Data Synthesis

Joining disparate clinical tables (Vitals, History, Complaints) into a unified, memory-optimized database using `patient_id` as the primary link.

In [ ]:
DATA_DIR = Path('/kaggle/input/triagegeist')
if not DATA_DIR.exists():
    # Local discovery engine for development
    DATA_DIR = Path('C:/Users/AMEY THAKUR/Downloads')

def load_and_merge(is_train=True):
    prefix = 'train' if is_train else 'test'
    
    df = pd.read_csv(DATA_DIR / f'{prefix}.csv')
    complaints = pd.read_csv(DATA_DIR / 'chief_complaints.csv')
    history = pd.read_csv(DATA_DIR / 'patient_history.csv')
    
    # Memory Optimization via Toolbox
    if tb: df = tb.reduce_mem_usage(df)
    
    # Relational Join: patient_id linkage binds profiles
    df = df.merge(complaints, on='patient_id', how='left')
    df = df.merge(history, on='patient_id', how='left')
    
    return df

train_raw = load_and_merge(is_train=True)
test_raw = load_and_merge(is_train=False)

print(f"Database Synthesis Complete. Clinical cohort: {len(train_raw):,} records.")
train_raw.head(3)

## 4. Exploratory Clinical Data Analysis (ECDA)

Visualizing class distributions and physiological volatility to identify clinical regimes of high variance. Understanding the clinical context of missingness is key to triage modelling.

In [ ]:
sns.set_palette('magma')
plt.style.use('bmh')

fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Target distribution: Assessing triage class balance
sns.countplot(x=CFG.TARGET, data=train_raw, ax=axes[0], palette='viridis')
axes[0].set_title('ESI Level Distribution: Triage Frequency')

# Vitals vs Acuity: Assessing signal strength in NEWS2
if 'news2_score' in train_raw.columns:
    sns.boxplot(x=CFG.TARGET, y='news2_score', data=train_raw, ax=axes[1], palette='magma')
    axes[1].set_title('NEWS2 Correlation with Triage Acuity')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(15, 6))
# Visualizing the "Missingness" Signal across variables
sns.heatmap(train_raw.isnull().T, cbar=False, xticklabels=False, cmap='viridis')
plt.title('Clinical Missingness Heatmap (Informative signals indicated in yellow)')
plt.show()

## 5. Strategic Feature Engineering

We implement the **Causal Guard** architecture to prevent data leakage. Informative missingness is explicitly captured via binary flags, and medical scoring systems like NEWS2 and Shock Index are prioritized.

In [ ]:
def build_clinical_features(df):
    # Enforcing causal guards: drop future artifacts (leaks)
    leakage_cols = ['ed_los_hours', 'disposition']
    df = df.drop(columns=[col for col in leakage_cols if col in df.columns])
    
    # Informative Missingness Mapping
    vital_v = ['systolic_bp', 'heart_rate', 'respiratory_rate', 'spo2', 'temperature_c']
    for col in vital_v:
        df[f'missing_{col}'] = df[col].isna().astype(int)
    df['vitals_missing_idx'] = df[vital_v].isnull().sum(axis=1)
    
    # Medical Signaling Logic
    if 'systolic_bp' in df.columns and 'diastolic_bp' in df.columns:
        # Pulse Pressure: Indicator of aortic stiffness or stroke volume
        df['pulse_pressure'] = df['systolic_bp'] - df['diastolic_bp']
        # Mean Arterial Pressure (MAP): Organ perfusion pressure
        df['map'] = df['diastolic_bp'] + (df['pulse_pressure'] / 3)
        
    # GCS Awareness (Level 1 prioritization)
    if 'gcs_total' in df.columns:
        df['is_unresponsive'] = (df['gcs_total'] <= 8).astype(int)
    
    # Type management for Category Handling
    cat_v = ['arrival_mode', 'arrival_day', 'sex', 'mental_status_triage']
    for col in cat_v:
        if col in df.columns: df[col] = df[col].astype('category')
            
    return df

train_fe = build_clinical_features(train_raw.copy())
test_fe = build_clinical_features(test_raw.copy())
print("Causal Feature Matrix ready. Clinical Indicators synthesized.")

## 6. Clinical NLP Synthesis (TF-IDF mapping)

Chief complaints provide the "qualitative intuition" in triage decisions. We utilize TF-IDF vectorized features to bridge the gap between structured vitals and clinical narrative.

In [ ]:
print("Generating TF-IDF clinical embeddings...")
tfidf = TfidfVectorizer(max_features=250, stop_words='english', ngram_range=(1, 2))

texts = train_fe['chief_complaint_raw'].fillna('unknown').values
test_texts = test_fe['chief_complaint_raw'].fillna('unknown').values

X_text_tr = tfidf.fit_transform(texts)
X_text_te = tfidf.transform(test_texts)

# Merge text components back as sparse meta-features
text_features = [f'text_{i}' for i in range(X_text_tr.shape[1])]
df_text_tr = pd.DataFrame(X_text_tr.toarray(), columns=text_features, index=train_fe.index)
df_text_te = pd.DataFrame(X_text_te.toarray(), columns=text_features, index=test_fe.index)

train_full = pd.concat([train_fe, df_text_tr], axis=1)
test_full = pd.concat([test_fe, df_text_te], axis=1)

print(f"NLP features synthesized: {len(text_features)} clinical concept features added.")

## 7. Multi-Tier Predictive Architecture

A hierarchical decision system combining deterministic heuristics with a diagnostic specialist layer and a balanced gradient-boosted ensemble.

In [ ]:
print("Initializing Tier 1: Deterministic Recognition")
# Mapping unambiguous clinical labels from training pattern memory
unambig = train_full.groupby('chief_complaint_raw')[CFG.TARGET].nunique()
unambig = unambig[unambig == 1].index

pattern_memory = train_full[train_full['chief_complaint_raw'].isin(unambig)].groupby(
    'chief_complaint_raw')[CFG.TARGET].first().to_dict()

print(f"Tier 1 Recognition: {len(pattern_memory):,} clinical patterns mapped.")

In [ ]:
print("Initializing Tier 3: Balanced Generalist Integration (LightGBM K-Fold)")
skip_cols = ['patient_id', 'chief_complaint_raw', CFG.TARGET]
f_cols = [c for c in train_full.columns if c not in skip_cols]

lgbm_params = {
    'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_error',
    'n_estimators': 1200, 'learning_rate': 0.04, 'num_leaves': 31,
    'class_weight': 'balanced', 'random_state': CFG.SEED, 'verbose': -1
}

skf = StratifiedKFold(n_splits=CFG.N_FOLDS, shuffle=True, random_state=CFG.SEED)
fold_bank = []
oof_preds = np.zeros((len(train_full), 5))

for fold, (tr_idx, val_idx) in enumerate(skf.split(train_full, train_full[CFG.TARGET]), 1):
    x_tr, y_tr = train_full.iloc[tr_idx][f_cols], train_full.iloc[tr_idx][CFG.TARGET] - 1
    x_vl, y_vl = train_full.iloc[val_idx][f_cols], train_full.iloc[val_idx][CFG.TARGET] - 1
    
    clf = lgb.LGBMClassifier(**lgbm_params)
    clf.fit(x_tr, y_tr)
    fold_bank.append(clf)
    oof_preds[val_idx] = clf.predict_proba(x_vl)
    print(f"Fold {fold}: Model calibrated.")

## 8. Clinical Error Analysis & Insight Dashboard

Analyzing where the system fails to match clinical priority. We focus on dangerous under-triage cases as our primary quality metric.

In [ ]:
oof_final = np.argmax(oof_preds, axis=1) + 1
y_true = train_full[CFG.TARGET].values

print("Clinical Performance Report (Generalist Fallback):")
print(classification_report(y_true, oof_final, target_names=list(CFG.ESI_LBLS.values())))

# Confusion Matrix Visualization for under-triage risk assessment
plt.figure(figsize=(10, 8))
cm = confusion_matrix(y_true, oof_final)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CFG.ESI_LBLS.keys(), yticklabels=CFG.ESI_LBLS.keys())
plt.title('Clinical Decision Heatmap (Audit for Under-triage Risk)')
plt.ylabel('Ground Truth (Clinician)')
plt.xlabel('Predicted Acuity (CDSS)')
plt.show()

## 9. Final Operational Synthesis

Synthesizing tiered decisions into a unified primary prediction stream.

In [ ]:
print("Step 9: Executing synthesis logic on inference cohort...")
ensemble_p = np.mean([f.predict_proba(test_full[f_cols]) for f in fold_bank], axis=0)
lgbm_res = np.argmax(ensemble_p, axis=1) + 1

test_final = []
for idx, txt in enumerate(test_full['chief_complaint_raw']):
    # Overriding with Tier 1 Pattern Memory if categorical match exists
    if txt in pattern_memory:
        test_final.append(pattern_memory[txt])
    else:
        test_final.append(lgbm_res[idx])

submission = pd.DataFrame({'patient_id': test_full['patient_id'], 'triage_acuity': test_final})
submission.to_csv('submission.csv', index=False)
print(f"Final Dataset Exported. Export Rows: {len(submission):,}")

***

## **10. Research Summary**

This research implemented a **Multi-Tier Clinical Decision Support System** to predict emergency triage acuity levels (ESI 1-5).

### Key Methodology:
1. **Clinical Narrative Alignment:** Leveraged **TF-IDF vectorization** to translate free-text chief complaints into sparse quantitative signals.
2. **Causal Integrity:** Enforced strict Guards to prevent look-ahead bias from future hospital metadata.
3. **Hemodynamic Feature Science:** Synthesized derived features like **Mean Arterial Pressure (MAP)** and **Pulse Pressure** consistent with clinical practice.
4. **Class Protection Modeling:** Used balanced LightGBM ensembles to preserve sensitivity for critical ESI-1/2 cases.

**Citation:** Olaf Yunus Laitinen Imanov. Triagegeist. https://www.kaggle.com/competitions/triagegeist. 2026. Kaggle.